## Sequence Purchasing Planning/Brainstorming

Lead     : `<Alex / AlexLeonardos>`

Issue    : [Github Issue #84](https://github.com/petadex/igem-toronto/issues/) — _Algorithm Planning_

Start    : `2026-06-01`


## Background and Motivation

We want to optimize how we purchase DNA sequences by considering the value of Degenerate-Codon Oligo Libraries. We can purchase a sequence where at specific positions, different bases can occur at a ratio we define. Using this, we could test many similar sequences from a petadex 60pid cluster even though we only purchased 1 sequence. Therefore, our motivation is to find the best sequence(s) to purchase for our needs, considering the following factors: 

1. **Sequence Quantity**: How many sequences are ordered.

    - Need some higher-level analysis to select which *families* to select from.
    - Could also run the algorithm on every family?? Depends on how expensive it is.

2. **Sequence Degeneracy**: How many degenerate bases we will allow in any given sequence.

    - Want to *minimize junk*, critical part of Artem's specification.

3. **Sequence Coverage**: How many natural amino acid sequences (exist in the chosen families) these purchased DNA sequences cover.

4. **RNA Structure**: Whether the RNA structure would be feasible (does it hairpin?)

    - Assign *-inf* to structures that aren't feasible. 
    - Computed using Claire's *scoring function*.

5. **Codon Ratios**: Which ratio of bases at these positions would maximize the amount of natural sequences synthesized and minimize the amount of artificial ones that aren't biologically existing.

    - Computed using Lisa's *scoring function*. This considers the codon table for specific vectors which assays would take place using.
    - Max 4 custom ratio bases according to Twist, although we may be able to pay for more
        - IUPAC Standard 50/50 combinations don't count towards this custom ratio count


## Problem Statement

We want to create an overall scoring function and algorithm that maximizes this function in order to choose the best sequences to order. 

## Section - Previous Solutions Summary

Claude Summary (*I've italicized my thoughts on the summaries and how they relate to our problem.*):

What the prior art actually does:

SwiftLib (Jacobs et al. 2015) — DP, position-independent.
This is exactly the multiplicative/separable structure I described. DP state = smallest library achieving error e using positions 1..i; the recursion multiplies library sizes and sums per-position error. O(n²m²), runs in ~0.6s.

- *This is quite fast, should probably look into maybe just using this algo if its conditions are similar to ours.*

Its objective is not "minimize junk" directly — junk is controlled only by a hard cap on total library size L = ∏ᵢ |dᵢ|. Error = sum of omitted observed-AA counts per column.

Hard limitation, stated explicitly: it cannot model covariation. Optimizing residue-pair information is what makes it NP-complete, so they deliberately assume column independence. 
It also freely packs in unobserved AAs (junk) to economize on DNA. This is the price of separability.

- *Think about these limitations, we do need to minimize junk, but I think that column independence is fine? Not sure what we could do with modelling/taking advantage of the columns if they're dependent on each other.*

- *Note: This is useful for avoiding junk. If two positions throughout the library are highly correlated, it'd be better to encode for the difference using separate oligo sequences, rather than degeneracy which introduces 50% junk.*

DeCoDe (Sumida et al. 2020) — ILP, position-dependent.

This one tackles the non-separable, conjunctive coverage head-on. It has a variable X_ij = "target sequence i is fully encoded by template j" (all positions must match — your conjunction), and maximizes the count of covered targets subject to a log-library-size budget.

It already solves two of your "missing" sub-problems:
Non-constant length → gap codons. Templates can use gap codons to produce shorter proteins; demonstrated on avGFP (238 vs 239 aa). So length variation is handled by gaps in the template, not a separate mechanism.

- *I think these gaps could solve some of the issues with non-constant length, but there is pretty large variation in our families*

Multiple oligos → "sublibraries." Group similar sequences, one degenerate template each, combine under a shared budget. This is the partitioning idea — but the ILP finds the grouping implicitly, which is exactly why it's expensive (hours to days; 48h cap with optimality gaps; scales exponentially in #sublibraries).
On NP-hardness: DeCoDe claims it via reduction from protein design (Pierce & Winfree 2002), not from clique. Your "reduction from clique" may be a different/independent argument — worth pinning down which, since it affects which approximation results you can borrow. I didn't find a clique reduction in these papers.
Notably: DeCoDe names ancestral reconstruction as a target use case — but it does not use phylogeny in the algorithm. Templates are free ILP variables.

- *Expand on this*
- *Our phylogenetic tree idea doesn't need to find the grouping manually, since phylogenetic trees do it for us.*
- *Is this necessarily better? Not sure, will determine from the benchmarks but the argument that you get both a phylogenetic tree and the oligo design from the algorithm is a potentially desirable one.*

DeCOIL (Yang et al. 2023) — stochastic search, fitness-guided.

Different objective entirely: it optimizes DC libraries to sample high-fitness and diverse variants for ML-assisted engineering, scored by a learned fitness predictor, via direct stochastic search. Relevant to you only if/when you want a "predicted-good" weight rather than pure natural-sequence coverage — but it shows the third algorithmic family (neither DP nor ILP: score-guided local search).

- *interesting to hear, maybe we could pivot to/expand on this once we have a ML model that can better predict plastic degredation ability/metadata.*

## Section: Brainstorming - Functions

### 1. *Compute Coverage* Function: def compute_coverage(sequence, family) -> int

The main idea of this function is to see how many sequences can be covered by an oligo.

Can't compute all possible proteins and check if the $N$ real sequences of length $L$ are in this set - set grows exponentially and is predominantly trash (hopefully not but that's the next section). 

Instead, loop over the real sequences and then for each amino acid, search if it can be encoded by the codon at the corresponding position. 

Can also avoid more complex data structures and search logic by encoding the genetic code with degenerate codons:

1. Store a dictionary of the genetic code (one for either direction).
2. Can create a dictionary for all of the amino-acids encoded by every degenerate codon, as there are $15^3$ of them. This is computed using the result of step 1.
3. Create $A_{c}$ list, where index $i$ contains the set of amino acids that can be encoded by the degenerate codon at position i, using the result from step 2. This will be a length 20-bitmap, where amino acid $a$ will consistently be at bit position $i_a$.
4. Loop over positions:
    a. Use $A_{c}[i]$ for every position $i$.
    b, $(A_c[i] >> i_a) \& 1$: Shifts the bit that represents membership of the AA of interest to the right, returns 1 if the bit is 1.
5. Early return if any are impossible.


This will run in worst case $O(N * L)$.

Could be further optimized with some preprocessing work to determine if some columns aren't decision columns, and therefore don't need to be recomputed every time because there's no variance.

Also need to consider what to do with columns with gaps -- more discussion on this later.


### 2. *Minimize Junk* Function: def min_junk(degen_bases, ratios) -> float

There's 2 types of junk, Coverage Junk $J_c$ and Ratio Junk $J_r$.

Let $C \in \mathbb{N}$ represent the total sequence space encoded for by the degenerate codons. 

$C = \prod_{i} d_i$, where $d_i$ is the number of amino acids producible at codon position c. 

Let $N \in \mathbb{N}$ be the number of real sequences that are reached by the oligo sequence.

Intuitively: $J_c = P - N$.

**Note:** Minimizing coverage junk is equivalent to maximizing coverage, which is why it may not be useful as a junk metric within the algorithm. I am recording this though in case we ultimately want to choose which sequences from which libraries to choose, as we can't send a sequence for each library to wet lab. $J_c + J_r$ would be a good way to compare the solutions between libraries as a total junk cost.

To define $J_r$, 

We need to formalize this better:

F = {x⁽¹⁾,…,x⁽ᴹ⁾}, with c codon positions. A candidate oligo O assigns to each codon position c three base distributions (p_{c,1}, p_{c,2}, p_{c,3}) over {A,C,G,T}.

Per-position amino-acid distribution (this is where the genetic code enters):

$P_c(a) = Σ_{(b₁b₂b₃) : code(b₁b₂b₃)=a}  p_{c,1}(b₁)·p_{c,2}(b₂)·p_{c,3}(b₃)$



NOTE: Lisa's organism encoding score could also play into this.

#### Formalizing `min_junk`

**Setup.** Let the target family be a set of aligned amino-acid sequences

$$
\mathcal{F} = \{x^{(1)}, \dots, x^{(M)}\}, \qquad x^{(i)} = (x^{(i)}_1, \dots, x^{(i)}_C),
$$

over $C$ codon positions. A candidate degenerate oligo $O$ assigns to each codon position $c$ three independent base distributions over the alphabet $\{A,C,G,T\}$:

$$
\big(p_{c,1}, \; p_{c,2}, \; p_{c,3}\big), \qquad \sum_{b \in \{A,C,G,T\}} p_{c,j}(b) = 1 .
$$

**Per-position amino-acid distribution.** The genetic code enters here. The probability that position $c$ produces amino acid $a$ (or a stop) is

$$
P_c(a) \;=\; \sum_{\substack{(b_1 b_2 b_3) \\ \text{code}(b_1 b_2 b_3) = a}} p_{c,1}(b_1)\, p_{c,2}(b_2)\, p_{c,3}(b_3),
\qquad a \in \mathcal{A}_{20} \cup \{\text{stop}\}.
$$

Stop mass $P_c(\text{stop})$ is junk automatically (truncation) — this is why $\text{NNK}$ is preferred over $\text{NNN}$.

**Synthesis distribution (always a product).** Mixed-base synthesis draws each position independently, so the distribution over produced proteins factorizes:

$$
Q_O(x) \;=\; \prod_{c=1}^{C} P_c(x_c).
$$

**Coverage mass** (covariation-aware — the product is summed *only over real target members*):

$$
\mathrm{Cov}(O) \;=\; \sum_{x \in \mathcal{F}} Q_O(x) \;=\; \sum_{x \in \mathcal{F}} \prod_{c=1}^{C} P_c(x_c).
$$

**Junk objective.**

$$
\boxed{\; J(O) \;=\; 1 - \mathrm{Cov}(O) \;=\; 1 - \sum_{x \in \mathcal{F}} \prod_{c=1}^{C} P_c(x_c) \;\in\; [0,1]. \;}
$$

A single number that absorbs every junk source: stop codons, cross-product **covariation leakage**, and mass misallocated to rare variants.

---

#### Division of labor

Because $Q_O$ is forced to be a **product** distribution, it can match only the *marginal* per-position frequencies — never correlations. Hence:

- **Custom ratios** $\Rightarrow$ remove *marginal* junk (mass bled onto rare variants at a single position).
- **Splitting into multiple oligos** $\Rightarrow$ remove *covariation* junk (e.g. the $\{VT, IS\}$ cross-product that also produces $VS, IT$).

#### Twist custom-ratio budget

Unlimited IUPAC positions (constrained to **uniform** mixes over a base subset: $R, V, N, \dots$) plus at most **4 custom** positions with arbitrary continuous ratios. 

Idea: Use greedy allocation and spend the 4 custom postiions on the positions whose family marginal is *most skewed*.

$$
\Delta_c \;=\; J_{\text{IUPAC}}(c) - J_{\text{custom}}(c), \qquad \text{pick } \arg\text{top-4}\; \Delta_c .
$$

#### Caveat: minimize subject to fixed support

Minimizing $J$ alone is degenerate — collapsing every position to one amino acid gives $\mathrm{Cov}=1$ but covers a single sequence. So $J$ is minimized **subject to a fixed support** $S_c = \{a : P_c(a) > 0\}$ set by the coverage step, with $p_{c,j}(b) > 0$ kept for every chosen base. With the support fixed the objective is interior; coordinate ascent over the $\le 4$ small simplices converges quickly.

```python
def junk_fraction(oligo, family) -> float:
    """J(O) = 1 - sum_{x in family} prod_c P_c(x_c).  Pure evaluation."""

def optimize_ratios(structure, family, custom_budget=4) -> tuple["Ratios", float]:
    """Given fixed support, choose <=4 custom-ratio positions + continuous
    ratios minimizing junk_fraction. Returns (ratios, achieved_junk)."""
```


### 3. How to deal with Families where there is non-constant sequence length:


*Baseline: Huge issue since the oligo can't synthesize sequences of different lengths.*
- unless I missed something??

1. QC - need insight on if my SQL approach is flawed and what these results could actually mean.

2. Need to perform alignment to be able to identify the locations and ranges of gaps.

    - Could fully align or just center around conserved catalytic core?

3. Split by length in ranges that vary and treat empty columns with gaps?

    - Run the algorithm on ranges rather than the full ORF.
    - would need to order more oligos to span gaps
    - Combine optimal solutions of the ranges, which should find the optimal solution?

 

### 4. *Phylogenetic Tree* Approximation (DP):

This approach will use the phylogenetic tree as a starting point.
This is useful since each node has two children, allowing for propagation of solutions through
a *DP* array using the following Bellman equation. 

Let $DP[r]$ represent the optimal score of the objective function and the corresponding sequence for this solution for a tree rooted at node $r$. Note that the root node r has two children in the phylogenetic tree, $L$ and $R$. 

Define *introduce_degen(L, R) -> sequence* to be the function to introduce degeneracy to create a sequence which can produce both of its children's sequences.

$DP[r][0] = max[DP[L][0], DP[R][0], \text{scoring\_func}(\text{introduce\_degen}(L, R))]$

$DP[r][1] =$ argmax sequence from the above equation.


Also needs a budget column, think about how to implement this.

### 5. *Phylogenetic Tree + Ancestral Reconstruction* Approximation (Greedy):

This approach will traverse the ancestral reconstruction tree from the root outwards, adding degeneracy in the most opportune positions using the reconstructions as a base.

Similar to greedy algo in essence, this won't actually get the best coverage but it'll get a pretty good ancestral reconstruction - based oligo which could be fine for a first pass?

Talk more with Thomas about this.